In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

# Spark is pre-initialized on Databricks as `spark`
print(f"Spark version: {spark.version}")
print(f"Current database: {spark.catalog.currentDatabase()}")

Spark version: 4.1.0
Current database: default


In [0]:
INPUT_CSV_PATH = "/Volumes/workspace/default/flight_data/ny_flights_phase3_bronze.csv"

BRONZE_TABLE = "bronze_flights"

print(f"Input CSV:    {INPUT_CSV_PATH}")
print(f"Bronze table: {BRONZE_TABLE}")

Input CSV:    /Volumes/workspace/default/flight_data/ny_flights_phase3_bronze.csv
Bronze table: bronze_flights


In [0]:
bronze_schema = StructType([
    StructField("Reporting_Airline", StringType(),  True),
    StructField("Origin",            StringType(),  True),
    StructField("Dest",              StringType(),  True),
    StructField("Route",             StringType(),  True),
    StructField("ArrDel15",          DoubleType(),  True),  # 0.0 / 1.0 — target
    StructField("Day",               IntegerType(), True),
    StructField("Month",             IntegerType(), True),
    StructField("DayOfWeek",         IntegerType(), True),
    StructField("IsWeekend",         IntegerType(), True),
    StructField("IsFixedHoliday",    IntegerType(), True),
    StructField("IsHolidayWindow",   IntegerType(), True),
    StructField("DepHour",           IntegerType(), True),
    StructField("DepTimeCategory",   StringType(),  True),
])

print("Schema defined with", len(bronze_schema.fields), "columns.")

Schema defined with 13 columns.


In [0]:
raw_df = (
    spark.read
         .option("header", "true")
         .schema(bronze_schema)
         .csv(INPUT_CSV_PATH)
)

print(f"Rows read: {raw_df.count():,}")
raw_df.printSchema()

Rows read: 309,108
root
 |-- Reporting_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Route: string (nullable = true)
 |-- ArrDel15: double (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- IsWeekend: integer (nullable = true)
 |-- IsFixedHoliday: integer (nullable = true)
 |-- IsHolidayWindow: integer (nullable = true)
 |-- DepHour: integer (nullable = true)
 |-- DepTimeCategory: string (nullable = true)



In [0]:
raw_df.show(5, truncate=False)

+-----------------+------+----+-------+--------+---+-----+---------+---------+--------------+---------------+-------+---------------+
|Reporting_Airline|Origin|Dest|Route  |ArrDel15|Day|Month|DayOfWeek|IsWeekend|IsFixedHoliday|IsHolidayWindow|DepHour|DepTimeCategory|
+-----------------+------+----+-------+--------+---+-----+---------+---------+--------------+---------------+-------+---------------+
|AA               |JFK   |LAX |JFK_LAX|0.0     |1  |1    |3        |0        |1             |1              |6      |Morning        |
|AA               |JFK   |LAX |JFK_LAX|0.0     |2  |1    |4        |0        |0             |1              |6      |Morning        |
|AA               |JFK   |LAX |JFK_LAX|0.0     |3  |1    |5        |0        |0             |0              |6      |Morning        |
|AA               |JFK   |LAX |JFK_LAX|0.0     |4  |1    |6        |1        |0             |0              |7      |Morning        |
|AA               |JFK   |LAX |JFK_LAX|0.0     |5  |1    |7   

In [0]:
bronze_df = (
    raw_df
      .withColumn("_ingested_at", F.current_timestamp())
      .withColumn("_source_file", F.col("_metadata.file_path"))
)

bronze_df.printSchema()

root
 |-- Reporting_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Route: string (nullable = true)
 |-- ArrDel15: double (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- IsWeekend: integer (nullable = true)
 |-- IsFixedHoliday: integer (nullable = true)
 |-- IsHolidayWindow: integer (nullable = true)
 |-- DepHour: integer (nullable = true)
 |-- DepTimeCategory: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)



In [0]:
(
    bronze_df.write
             .format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(BRONZE_TABLE)
)

print(f"✓ Wrote Delta table: {BRONZE_TABLE}")

✓ Wrote Delta table: bronze_flights


In [0]:
row_count = spark.table(BRONZE_TABLE).count()
print(f"Bronze table row count: {row_count:,}")
print(f"Expected (from Phase 2): 309,108")
assert row_count > 300_000, "Row count looks too low — investigate."

Bronze table row count: 309,108
Expected (from Phase 2): 309,108


In [0]:
null_counts = spark.table(BRONZE_TABLE).select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in bronze_schema.fieldNames()
])
null_counts.show(truncate=False)

+-----------------+------+----+-----+--------+---+-----+---------+---------+--------------+---------------+-------+---------------+
|Reporting_Airline|Origin|Dest|Route|ArrDel15|Day|Month|DayOfWeek|IsWeekend|IsFixedHoliday|IsHolidayWindow|DepHour|DepTimeCategory|
+-----------------+------+----+-----+--------+---+-----+---------+---------+--------------+---------------+-------+---------------+
|0                |0     |0   |0    |0       |0  |0    |0        |0        |0             |0              |0      |0              |
+-----------------+------+----+-----+--------+---+-----+---------+---------+--------------+---------------+-------+---------------+



In [0]:
print("ArrDel15 distribution (0 = on-time, 1 = delayed >= 15 min):")
spark.sql(f"""
    SELECT ArrDel15,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM {BRONZE_TABLE}
    GROUP BY ArrDel15
    ORDER BY ArrDel15
""").show()

ArrDel15 distribution (0 = on-time, 1 = delayed >= 15 min):
+--------+------+-----+
|ArrDel15|     n|  pct|
+--------+------+-----+
|     0.0|234864|75.98|
|     1.0| 74244|24.02|
+--------+------+-----+



In [0]:
print("Airlines (Reporting_Airline):")
spark.sql(f"SELECT DISTINCT Reporting_Airline FROM {BRONZE_TABLE} ORDER BY 1").show(20, truncate=False)

print("NY Origin airports:")
spark.sql(f"""
    SELECT Origin, COUNT(*) AS n
    FROM {BRONZE_TABLE}
    GROUP BY Origin
    ORDER BY n DESC
""").show(20, truncate=False)

print("Departure time categories:")
spark.sql(f"SELECT DISTINCT DepTimeCategory FROM {BRONZE_TABLE}").show(truncate=False)

Airlines (Reporting_Airline):
+-----------------+
|Reporting_Airline|
+-----------------+
|9E               |
|AA               |
|AS               |
|B6               |
|DL               |
|F9               |
|G4               |
|HA               |
|MQ               |
|NK               |
|OH               |
|OO               |
|UA               |
|WN               |
|YX               |
+-----------------+

NY Origin airports:
+------+------+
|Origin|n     |
+------+------+
|LGA   |133234|
|JFK   |104024|
|BUF   |20338 |
|ALB   |12241 |
|HPN   |10587 |
|SYR   |10334 |
|ROC   |10046 |
|ISP   |5186  |
|ELM   |1215  |
|SWF   |603   |
|PBG   |539   |
|IAG   |464   |
|BGM   |234   |
|ITH   |63    |
+------+------+

Departure time categories:
+---------------+
|DepTimeCategory|
+---------------+
|Morning        |
|Evening        |
|Afternoon      |
|Late Night     |
+---------------+



In [0]:

spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}").show(5, truncate=False)

+-------+-------------------+--------------+---------------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------+
|version|timestamp          |userId        |userName                   |operation                        |operationParameters                                                                                                                                                                             |job |notebook          |queryHistoryStatementId             |clu

## 8. Summary

| Step | Status |
|---|---|
| Loaded 309K flight records with explicit schema | ✓ |
| Added lineage columns (`_ingested_at`, `_source_file`) | ✓ |
| Saved as Delta table `bronze_flights` | ✓ |
| Sanity checks passed | ✓ |

### Next Step

Proceed to **Notebook 02 (Silver Layer)** to perform data quality checks, type enforcement,
and deduplication.